# Prange ISD exploration (Strategy 02: ISD syndrome-decoding prototypes)

Early standalone exploration of plain Prange Information Set Decoding applied to the PEP-to-syndrome-decoding transformation, predating the main prediction-and-repair framework in `Code/core/`. Not actively used by later strategies - kept as a reference for how the syndrome-decoding reduction was first worked out.


In [ ]:
import os
import sys

# instances_generator.py / LEP_prediction_and_repair_v2.py now live in
# Code/core/ (this notebook was relocated during the Aug 2026 folder
# reorganization), so it needs to be added to sys.path explicitly. Tries
# a few relative depths so this works whether Jupyter's cwd is this
# notebook's own folder or the Code/ root.
for _rel in ['../../core', '../core', 'core']:
    if os.path.isdir(_rel):
        sys.path.insert(0, os.path.abspath(_rel))


In [1]:
from instances_generator import *
import random

n = 7  # length of the code
k = 3   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)
H_tilde_1, vectorP_noisy_1 = transform_problem_to_syndrome_decoding_H1(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

print("Generator matrix G1:")
print(G1)
print("\nGenerator matrix G2:")
print(G2)
print("\nSecret permutation matrix P:")
print(P)
print("\nNoisy hint for P:")
print(P_noisy)

print("\nParity-check matrix H_tilde:")
print(H_tilde)
print("\nNoisy hint vector for P:")
print(vectorP_noisy)

Generator matrix G1:
[1 0 0 3 0 5 1]
[0 1 0 3 6 4 6]
[0 0 1 6 4 6 1]

Generator matrix G2:
[1 0 0 3 0 2 3]
[0 1 0 2 2 4 1]
[0 0 1 1 2 2 6]

Secret permutation matrix P:
[0 0 0 0 0 0 1]
[0 1 0 0 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 1 0 0 0 0]
[1 0 0 0 0 0 0]
[0 0 0 1 0 0 0]

Noisy hint for P:
[0 0 0 0 0 0 0]
[0 1 0 0 0 0 0]
[0 0 0 0 1 0 0]
[0 0 0 0 0 1 0]
[0 0 1 0 0 0 0]
[1 0 0 0 0 0 0]
[0 0 0 1 0 0 0]

Parity-check matrix H_tilde:
[1 0 0 3 0 5 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 6 0 3 2 1 0 0 3 0 5 1 6 0 0 4 0 2 6]
[0 1 0 3 6 4 6 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 6 5 1 5 0 1 0 3 6 4 6 0 6 0 4 1 3 1]
[0 0 1 6 4 6 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 5 1 5 2 0 0 1 6 4 6 1 0 0 6 1 3 1 6]
[0 0 0 0 0 0 0 1 0 0 3 0 5 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 6 0 3 2 2 0 0 6 0 3 2 1 0 0 3 0 5 1]
[0 0 0 0 0 0 0 0 1 0 3 6 4 6 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 6 5 1 5 0 2 0 6 5 1 5 0 1 0 3 6 4 6]
[0 0 0 0 0 0 0 0 0 1 6 4 6 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [ ]:
def prange_decoder(H, s, w, F, max_trials=10000):
    """
    Prange Information Set Decoding.

    Input:
        H: parity-check matrix over F
        s: syndrome vector of length n-k
        w: target error weight
        max_trials: maximum number of trials

    Output:
        e_candidate, trial if found
        None, max_trials otherwise
    """

    H = Matrix(F, H)

    # If s is a column matrix, use s.list()
    if hasattr(s, "list"):
        s = vector(F, s.list())
    else:
        s = vector(F, s)

    n_minus_k, n = H.dimensions()
    k = n - n_minus_k

    for trial in range(1, max_trials + 1):

        # choose random information set I of size k
        I_indices = random.sample(range(n), k)

        # complement J
        I_set = set(I_indices)
        J_indices = [j for j in range(n) if j not in I_set]

        # permutation order
        P_indices = I_indices + J_indices

        # permutation matrix
        P = Matrix(F, n, n, 0)
        for new_idx, original_idx in enumerate(P_indices):
            P[original_idx, new_idx] = 1

        # H_prime = [H_I | H_J]
        H_prime = H * P

        H_I = H_prime.submatrix(0, 0, n_minus_k, k)
        H_J = H_prime.submatrix(0, k, n_minus_k, n_minus_k)

        if not H_J.is_invertible():
            continue

        # Solve H_J * e_J = s
        e_J = H_J.solve_right(s)

        # Assume error is zero on information set I
        e_permuted = vector(F, [0] * k + list(e_J))

        # Undo permutation
        e_candidate = vector(F, n)
        for new_idx, original_idx in enumerate(P_indices):
            e_candidate[original_idx] = e_permuted[new_idx]

        # Check weight
        if e_candidate.hamming_weight() != w:
            continue

        # Check syndrome
        if H * e_candidate.column() == s.column():
            return e_candidate, trial

    return None, max_trials

In [ ]:
F = GF(7)

vectorP_noisy = vector(F, vectorP_noisy)
vectorP = vector(F, vectorP)

vectorE = vectorP_noisy - vectorP
w = sum(1 for x in vectorE if x != 0)

w

3

In [ ]:
s  = H_tilde * vectorP_noisy
s1 = H_tilde_1 * vectorP_noisy_1

(2, 2, 2, 5, 6, 6, 2, 4, 4, 3, 2, 2, 3, 6, 5, 6, 1, 0, 0, 0, 3, 2, 4, 2)

In [ ]:
e, _ = prange_decoder(H_tilde, s2, w, F, 1000000)
e

(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0)

In [ ]:
vectorE == e

True

In [ ]:
n = 20  # length of the code
k = 10   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

F = GF(q)

vectorP_noisy = vector(F, vectorP_noisy)
vectorP = vector(F, vectorP)

vectorE = vectorP_noisy - vectorP
w = sum(1 for x in vectorE if x != 0)
s  = H_tilde * vectorP_noisy
e, _ = prange_decoder(H_tilde, s, w, F, 1000000)
vectorE == e

True

In [ ]:
n = 30  # length of the code
k = 15   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

F = GF(q)

vectorP_noisy = vector(F, vectorP_noisy)
vectorP = vector(F, vectorP)

vectorE = vectorP_noisy - vectorP
w = sum(1 for x in vectorE if x != 0)
s  = H_tilde * vectorP_noisy
e, _ = prange_decoder(H_tilde, s, w, F, 1000000)
vectorE == e

True

In [ ]:
n = 50  # length of the code
k = 25   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

F = GF(q)

vectorP_noisy = vector(F, vectorP_noisy)
vectorP = vector(F, vectorP)

vectorE = vectorP_noisy - vectorP
w = sum(1 for x in vectorE if x != 0)
s  = H_tilde * vectorP_noisy
e, _ = prange_decoder(H_tilde, s, w, F, 1000000)
vectorE == e

In [ ]:
n = 100  # length of the code
k = 50   # dimension of the code
q = 7   # size of the finite field
alpha = 0.01  # probability of flipping 0 to a random element
beta = 0.2   # probability of flipping a random element to 0

# Generate a noisy PEP instance
G1, G2, P, P_noisy = generate_noisy_LCE_instance_CBA(n, k, q, alpha, beta, is_monomial=False)

# Transform the problem to a syndrome decoding problem
H_tilde, vectorP_noisy = transform_problem_to_syndrome_decoding(G1, G2, P_noisy)

vectorP = transform_secret_to_single_vector(P)

F = GF(q)

vectorP_noisy = vector(F, vectorP_noisy)
vectorP = vector(F, vectorP)

vectorE = vectorP_noisy - vectorP
w = sum(1 for x in vectorE if x != 0)
s  = H_tilde * vectorP_noisy
e, _ = prange_decoder(H_tilde, s, w, F, 1000000)
vectorE == e